<a href="https://colab.research.google.com/github/LatiefDataVisionary/pm-tools-absa-analysis/blob/main/KDK_Scraping_Ulasan_dari_TrustRadius.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install playwright
!playwright install --with-deps chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 9.0 MB/s eta 0:00:00
Installing dependencies...
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [61.6 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,863 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe 

In [2]:
import pandas as pd
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import re

In [3]:
async def scrape_detailed_reviews(url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
        page = await context.new_page()

        page.set_default_timeout(60000)
        results = []

        try:
            print(f"Mengambil data detail dari: {url}")
            await page.goto(url, wait_until="domcontentloaded")
            await asyncio.sleep(7)

            html = await page.content()
            soup = BeautifulSoup(html, 'html.parser')

            product_name = soup.find('h1').get_text(strip=True) if soup.find('h1') else "Jira"
            review_cards = soup.select('div[data-testid="review-card"], article')

            for card in review_cards:
                # EKSTRAKSI RATING: Mencari aria-label yang berisi angka rating
                rating = "N/A"
                rating_elem = card.select_one('[class*="Rating_rating"]')
                if rating_elem and rating_elem.has_attr('aria-label'):
                    # Contoh: "Rated 9 out of 10" -> ambil angka 9
                    aria_text = rating_elem['aria-label']
                    rating_match = re.search(r'(\d+)', aria_text)
                    if rating_match:
                        rating = rating_match.group(1)

                # Ekstrak Judul Ulasan
                title_elem = card.find(['h3', 'h2'])
                review_title = title_elem.get_text(strip=True) if title_elem else "N/A"

                # Ekstrak Pros & Cons
                pros = ""
                cons = ""
                sections = card.find_all('div', class_=True)
                for sec in sections:
                    txt = sec.get_text(strip=True)
                    if "Pros" in txt and len(txt) < 500: pros = txt.replace("Pros", "").strip()
                    if "Cons" in txt and len(txt) < 500: cons = txt.replace("Cons", "").strip()

                results.append({
                    "product_name": product_name,
                    "review_title": review_title,
                    "rating": rating,
                    "pros": pros,
                    "cons": cons,
                    "full_content": card.get_text(strip=True)[:300] + "..."
                })

        except Exception as e:
            print(f"Error: {e}")
        finally:
            await browser.close()

        return results

In [4]:
async def scrape_multiple_pages(product_name, base_url, total_pages):
    all_page_data = []

    for i in range(1, total_pages + 1):
        url = f"{base_url}?page={i}"
        print(f"[{product_name}] Sedang memproses halaman {i}/{total_pages}")

        page_results = await scrape_detailed_reviews(url)

        if page_results:
            for item in page_results:
                item['page_number'] = i
                item['product_name'] = product_name
            all_page_data.extend(page_results)
        else:
            print(f"Peringatan: Tidak ada data di {product_name} halaman {i}")

    df = pd.DataFrame(all_page_data)
    filename = f"master_{product_name.lower()}_reviews.csv"
    df.to_csv(filename, index=False)
    print(f"Selesai! File '{filename}' telah disimpan dengan {len(df)} ulasan.\n")
    return df

In [5]:
# Daftar target scraping
targets = [
    {"name": "Jira", "url": "https://www.trustradius.com/products/atlassian-jira/reviews/all", "pages": 23},
    {"name": "Trello", "url": "https://www.trustradius.com/products/trello/reviews/all", "pages": 23},
    {"name": "ClickUp", "url": "https://www.trustradius.com/products/clickup/reviews/all", "pages": 72}
]

# Eksekusi berurutan
for target in targets:
    await scrape_multiple_pages(target['name'], target['url'], target['pages'])

print("Semua proses scraping selesai!")

[Jira] Sedang memproses halaman 1/23
Mengambil data detail dari: https://www.trustradius.com/products/atlassian-jira/reviews/all?page=1
[Jira] Sedang memproses halaman 2/23
Mengambil data detail dari: https://www.trustradius.com/products/atlassian-jira/reviews/all?page=2
[Jira] Sedang memproses halaman 3/23
Mengambil data detail dari: https://www.trustradius.com/products/atlassian-jira/reviews/all?page=3
[Jira] Sedang memproses halaman 4/23
Mengambil data detail dari: https://www.trustradius.com/products/atlassian-jira/reviews/all?page=4
[Jira] Sedang memproses halaman 5/23
Mengambil data detail dari: https://www.trustradius.com/products/atlassian-jira/reviews/all?page=5
[Jira] Sedang memproses halaman 6/23
Mengambil data detail dari: https://www.trustradius.com/products/atlassian-jira/reviews/all?page=6
[Jira] Sedang memproses halaman 7/23
Mengambil data detail dari: https://www.trustradius.com/products/atlassian-jira/reviews/all?page=7
[Jira] Sedang memproses halaman 8/23
Mengambil d